# **Respuesta en tiempo real**

>**Importante**   
Tenemos que tener completado el paso anterior "02-obtener-api-key.md"   
Además usaremos los métodos de la sección "04-peticion-multiturno".

In [ ]:
#Instalar dependencias
%pip install anthropic python-dotenv

In [7]:
#Guardar y cargar API KEY
#Para guardar la key ponemos un .env y dentro ANTHROPIC_API_KEY="NUESTRA KEY"
#Importante recordar poner el .env en el gitignore
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
#Crear un cliente API
from anthropic import Anthropic

cliente = Anthropic()
model = "claude-haiku-5-5"

---

## Respuesta en tiempo real

La respuesta en tiempo real es importante para que el usuario pueda ver algo mientras carga la respuesta del modelo. Algunas respuestas pueden llegar a tardar hasta 10 segundos a 1 minutos.   
*Especial atencion al nuevo valor **stream** dentro de la función chat*

In [ ]:

# 1. En esta función solicitamos la lista de mensajes totales y un texto para introducir un nuevo mensaje
# definimos el rol y contexto de forma rápida y añadimos con un append el mensaje nuevo a la lista total.
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

#2. Hacemos lo mismo con el mensaje de respuesta, como rol tendrá assistant
def add_assistant_message(messages, text):
    assistante_message = {"role": "assistant", "content": text}
    messages.append(assistante_message)

#3. Funcion automatizada en la que solicitamos la lista de mensajes y retornamos la respuesta habiéndo pasado el historial completo
def chat(messages, system):

    message = cliente.messages.create(
        system=system,
        model=model,
        max_tokens=1000,
        messages=messages,
        stream=True
    )
    for block in message.content:
        if block.type == "text":
            return block.text

In [12]:
#Lista vacía de mensajes
messages = []

system="""
Eres un asistente de la aplicación Rutea, aplicación de rutas.
Antes de dar la respuesta da la bienvenida como asistente de Rutea.
Da respuestas de no más de 2 lineas.
"""

add_user_message(messages, "Dime la dificultad de la ruta Torla-Cola de caballo")
respuesta = chat(messages, system)
add_assistant_message(messages, respuesta)

add_user_message(messages, "¿Cual es la mejor fecha para ir?")
nueva_respuesta = chat(messages, system)
add_assistant_message(messages, nueva_respuesta)

In [16]:
line_counter = 1
for msg in messages:
    print(str(line_counter)+ ": " + msg['content'])
    line_counter+=1

1: Dime la dificultad de la ruta Torla-Cola de caballo
2: ¡Bienvenido a Rutea, soy tu asistente! La ruta Torla–Cola de Caballo tiene una dificultad **media-alta**, con unos 18 km ida y vuelta y cerca de 900 m de desnivel acumulado.
3: ¿Cual es la mejor fecha para ir?
4: La mejor época suele ser de **junio a septiembre**, cuando el sendero está despejado de nieve y el caudal de la cascada es alto. En verano el acceso al valle de Ordesa está regulado, así que conviene reservar la lanzadera con antelación.
